In [46]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

import lsst.geom
from lsst.daf.butler import Butler
from lsst.rsp import get_tap_service

In [47]:
rsp_tap = get_tap_service("tap")
assert rsp_tap is not None

In [48]:
butler = Butler('dp1', collections='LSSTComCam/DP1')
assert butler is not None

In [ ]:
# Sampling parameters
START_NIGHT = 1          # first night index to include (1 to 48, inclusive)
END_NIGHT = 4           # last night index to include (1 to 48, inclusive)
VISITS_PER_NIGHT = 5     # number of visits to sample per night
STARS_PER_VISIT = 40     # number of stars to sample per visit
DETECTOR_ID = 0          # detector to use

In [ ]:
# Step 1: Query all available nights (day_obs) across the full dataset,
# then select the subset indexed by START_NIGHT to END_NIGHT (1-based, chronological order)
from collections import defaultdict

all_visit_refs = list(butler.query_datasets(
    'visit_image',
    where=f"detector = {DETECTOR_ID}",
    order_by=['day_obs', 'visit'],
    with_dimension_records=True,
))

# Group references by night (day_obs)
night_to_refs = defaultdict(list)
for ref in all_visit_refs:
    night_to_refs[ref.dataId['day_obs']].append(ref)

all_nights_sorted = sorted(night_to_refs.keys())
print(f"Total nights available: {len(all_nights_sorted)}")

# Select the range [START_NIGHT, END_NIGHT] (1-indexed)
selected_nights = all_nights_sorted[START_NIGHT - 1 : END_NIGHT]
print(f"Selected nights {START_NIGHT}–{END_NIGHT}: {len(selected_nights)} nights "
      f"(day_obs {selected_nights[0]} to {selected_nights[-1]})")

In [ ]:
# Step 2: Sample up to VISITS_PER_NIGHT visits from each selected night (detector fixed to DETECTOR_ID)
import random

random.seed(42)  # for reproducibility; change or remove to vary the sample

selected_visit_refs = []
for night in selected_nights:
    night_refs = night_to_refs[night]
    n_sample = min(VISITS_PER_NIGHT, len(night_refs))
    sampled = random.sample(night_refs, n_sample)
    selected_visit_refs.extend(sampled)

visit_ids = [ref.dataId['visit'] for ref in selected_visit_refs]
print(f"Total visits selected: {len(selected_visit_refs)}")
print(f"Visit IDs: {visit_ids}")

In [ ]:
# Step 3: Query dp1.Source for PSF-used stars across all selected visits (one batched TAP query),
# then randomly sample STARS_PER_VISIT stars from each visit

visit_id_list = ', '.join(str(v) for v in visit_ids)
query = (
    "SELECT sourceId, x, y, visit "
    "FROM dp1.Source "
    f"WHERE visit IN ({visit_id_list}) "
    f"AND detector = {DETECTOR_ID} "
    "AND calib_psf_used = 1"
)

job = rsp_tap.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

all_stars_table = job.fetch_result().to_table()
print(f"Found {len(all_stars_table)} PSF candidate stars across {len(selected_visit_refs)} visits")

# Group stars by visit and randomly sample STARS_PER_VISIT from each
rng = np.random.default_rng(seed=42)  # for reproducibility; change seed to vary

visit_star_tables = {}
for ref in selected_visit_refs:
    vid = ref.dataId['visit']
    mask = all_stars_table['visit'] == vid
    visit_stars = all_stars_table[mask]
    if len(visit_stars) == 0:
        print(f"  Visit {vid}: no calib_psf_used stars found, will be skipped")
        continue
    n_sample = min(STARS_PER_VISIT, len(visit_stars))
    chosen = rng.choice(len(visit_stars), size=n_sample, replace=False)
    visit_star_tables[vid] = visit_stars[chosen]
    print(f"  Visit {vid}: {len(visit_stars)} available, {n_sample} sampled")

In [53]:
import sys
from pathlib import Path

for src_path in (Path.cwd() / 'src', Path.cwd().parent / 'src'):
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))
        break

import importlib
import fittingTools
importlib.reload(fittingTools)


<module 'fittingTools' from '/Users/jinshuozhang/Library/CloudStorage/Dropbox/Rubin-Work/Rubin-PSF-Analysis/src/fittingTools.py'>

In [ ]:
# Step 4: For each selected visit, load visit_image and run PSF model comparison analysis.
# The per-star analysis logic is unchanged; this cell loops over all loaded visits.

for visit_ref in selected_visit_refs:
    visit_id = visit_ref.dataId['visit']
    detector_id = visit_ref.dataId['detector']
    band = visit_ref.dataId['band']

    if visit_id not in visit_star_tables:
        print(f"\nVisit {visit_id}: skipping (no stars)")
        continue

    psf_used_table = visit_star_tables[visit_id]
    print(f"\n--- Visit {visit_id}, Detector {detector_id}, Band {band} "
          f"({len(psf_used_table)} stars) ---")

    visit_image = butler.get(visit_ref)
    psf = visit_image.getPsf()

    reference_position = lsst.geom.Point2D(300, 300)
    reference_stamp_dims = psf.computeImage(reference_position).getDimensions()
    print(f"PSF stamp size: {reference_stamp_dims} pix")

    # --- Route B: extract observed star cutouts and compare models ---
    n_stars = len(psf_used_table)

    print('Route B analysis chain: Butler -> visit_image -> getCutout(position, stamp size) -> observed star array')

    t0 = time.time()
    results = []
    n_skipped = 0
    for i in range(n_stars):
        try:
            x_star = psf_used_table['x'][i]
            y_star = psf_used_table['y'][i]
            position_star = lsst.geom.Point2D(x_star, y_star)

            rubin_psf_model = psf.computeImage(position_star)
            stamp_dims = rubin_psf_model.getDimensions()
            xlen = stamp_dims.getX()
            ylen = stamp_dims.getY()

            star_cutout = visit_image.getCutout(position_star, lsst.geom.Extent2I(xlen, ylen))
            star_masked_image = star_cutout.getMaskedImage()
            star_image_array = star_masked_image.image.array.astype(float)

            star_flux = np.sum(star_image_array)
            if not np.isfinite(star_flux) or star_flux <= 0:
                raise ValueError('non-positive observed cutout flux')

            star_image_array = star_image_array / star_flux

            psf_shape = psf.computeShape(position_star)
            psf_sigma = psf_shape.getDeterminantRadius()
            psf_fwhm = psf_sigma * fittingTools.SIGMA_TO_FWHM

            result = fittingTools.analyze_psf_models(
                star_image_array,
                x=x_star,
                y=y_star,
                fwhm=psf_fwhm,
            )
            result['rubin_psf_array'] = rubin_psf_model.array / np.sum(rubin_psf_model.array)
            results.append(result)

        except Exception as e:
            n_skipped += 1
            print('Star %d skipped: %s' % (i, e))

    t_compute = time.time() - t0
    print('Computation done: %d/%d stars in %.2fs (%d skipped)' % (
        len(results), n_stars, t_compute, n_skipped))

    # --- VISUALIZATION PHASE ---
    page_size = 3
    n_computed = len(results)

    t0 = time.time()
    fittingTools.plot_model_comparison_pages(
        results,
        page_size,
        visit_id,
        detector_id,
        band,
    )
    best_model_counts = fittingTools.plot_best_model_counts(results)
    t_plot = time.time() - t0
    print('Plotting done: %d stars in %.2fs (%d per page)' % (n_computed, t_plot, page_size))
    print('Best model counts:', best_model_counts)